In [6]:
import open3d as o3d
import numpy as np

In [4]:
rgb = o3d.io.read_image("image1_rgb.jpg")
depth = o3d.io.read_image("image1_depth.png")
rgbd1 = o3d.geometry.RGBDImage.create_from_color_and_depth(
        rgb, depth, depth_trunc=4.0, convert_rgb_to_intensity=False)

In [5]:
rgb = o3d.io.read_image("image2_rgb.jpg")
depth = o3d.io.read_image("image2_depth.png")
rgbd2 = o3d.geometry.RGBDImage.create_from_color_and_depth(
        rgb, depth, depth_trunc=4.0, convert_rgb_to_intensity=False)

In [8]:
pinhole_camera_intrinsic = o3d.camera.PinholeCameraIntrinsic(
    width=640, height=480, fx=525, fy=525, cx=319.5, cy=239.5
)
pinhole_camera_intrinsic.intrinsic_matrix

array([[525. ,   0. , 319.5],
       [  0. , 525. , 239.5],
       [  0. ,   0. ,   1. ]])

In [33]:
option = o3d.pipelines.odometry.OdometryOption()
odo_init = np.identity(4)
print(option)

[success_color_term, trans_color_term,
 info] = o3d.pipelines.odometry.compute_rgbd_odometry(
     rgbd2, rgbd1, pinhole_camera_intrinsic, odo_init,
     o3d.pipelines.odometry.RGBDOdometryJacobianFromColorTerm(), option)
print(trans_color_term)

[success_hybrid_term, trans_hybrid_term,
 info] = o3d.pipelines.odometry.compute_rgbd_odometry(
     rgbd2, rgbd1, pinhole_camera_intrinsic, odo_init,
     o3d.pipelines.odometry.RGBDOdometryJacobianFromHybridTerm(), option)
print(trans_hybrid_term)

OdometryOption class.
iteration_number_per_pyramid_level = [ 20, 10, 5, ] 
max_depth_diff = 0.030000
min_depth = 0.000000
max_depth = 4.000000
[[ 0.99997933  0.00617403  0.001796    0.01200112]
 [-0.0061877   0.99995112  0.00771217  0.00862762]
 [-0.00174829 -0.00772312  0.99996865  0.00540695]
 [ 0.          0.          0.          1.        ]]
[[ 9.99979187e-01  6.45101534e-03 -1.03706031e-04  1.59085285e-02]
 [-6.45037933e-03  9.99965311e-01  5.26960813e-03  1.50154433e-02]
 [ 1.37696756e-04 -5.26882951e-03  9.99986110e-01  1.07213473e-02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


numpy.ndarray

In [21]:
pcd1 = o3d.geometry.PointCloud.create_from_rgbd_image(
    rgbd1, pinhole_camera_intrinsic)

In [30]:
pcd2 = o3d.geometry.PointCloud.create_from_rgbd_image(
    rgbd2, pinhole_camera_intrinsic)

pcd2.transform(trans_color_term)
print(pcd2)
o3d.visualization.draw_geometries([pcd1, pcd2],
                                    zoom=0.48,
                                    front=[0.0999, -0.1787, -0.9788],
                                    lookat=[0.0345, -0.0937, 1.8033],
                                    up=[-0.0067, -0.9838, 0.1790])

PointCloud with 180782 points.


In [31]:
pcd2 = o3d.geometry.PointCloud.create_from_rgbd_image(
    rgbd2, pinhole_camera_intrinsic)

pcd2.transform(trans_hybrid_term)
o3d.visualization.draw_geometries([pcd1, pcd2],
                                    zoom=0.48,
                                    front=[0.0999, -0.1787, -0.9788],
                                    lookat=[0.0345, -0.0937, 1.8033],
                                    up=[-0.0067, -0.9838, 0.1790])

In [32]:
volume = o3d.pipelines.integration.ScalableTSDFVolume(
    voxel_length=4.0 / 512.0,
    sdf_trunc=0.04,
    color_type=o3d.pipelines.integration.TSDFVolumeColorType.RGB8)

In [35]:
pose0 = np.array([
    [1,0,0,0],
    [0,1,0,0],
    [0,0,1,0],
    [0,0,0,1]
])

In [36]:
volume.integrate(
    rgbd1,
    o3d.camera.PinholeCameraIntrinsic(o3d.camera.PinholeCameraIntrinsicParameters.PrimeSenseDefault),
    np.linalg.inv(pose0)
)

In [37]:
volume.integrate(
    rgbd2,
    o3d.camera.PinholeCameraIntrinsic(o3d.camera.PinholeCameraIntrinsicParameters.PrimeSenseDefault),
    np.linalg.inv(trans_hybrid_term)
)

In [41]:
mesh = volume.extract_triangle_mesh()
mesh.compute_vertex_normals()
o3d.visualization.draw_geometries([mesh],
                                zoom=0.48,
                                front=[0.0999, -0.1787, -0.9788],
                                lookat=[0.0345, -0.0937, 1.8033],
                                up=[-0.0067, -0.9838, 0.1790])